# dbAppsUA07: SQL JOIN
## Застосування баз даних

Цей урок покриває основні типи SQL JOIN:
- **INNER JOIN** — тільки рядки що співпадають в обох таблицях
- **LEFT JOIN** — всі рядки з лівої таблиці + співпадіння з правої
- **Багатотабличні JOIN** — з'єднання 3+ таблиць

## Налаштування

In [ ]:
import pandas as pd
import sqlite3

# Підключаємося до бази даних IMDb
conn = sqlite3.connect('imdb_class.db')

print("Підключення встановлено!")

## Розділ 1: INNER JOIN

**INNER JOIN** повертає тільки рядки де є співпадіння в **обох** таблицях.

```sql
SELECT columns
FROM table1
INNER JOIN table2
ON table1.key = table2.key
```

**ON** — вказує умову що з'єднує рядки двох таблиць.

In [ ]:
# Приклад 1: З'єднуємо фільми з їх рейтингами
# tb — псевдонім (alias) для title_basics
# tr — псевдонім для title_ratings
query = """
SELECT 
    tb.primaryTitle,
    tb.startYear,
    tr.averageRating,
    tr.numVotes
FROM title_basics tb
INNER JOIN title_ratings tr ON tb.tconst = tr.tconst
WHERE tb.titleType = 'movie'
LIMIT 10
"""

result = pd.read_sql(query, conn)
print("Фільми з рейтингами (INNER JOIN):")
print(result)

**Спробуй:** Напиши INNER JOIN запит що показує назву фільму та рейтинг для фільмів 2015-2020, відсортуй за рейтингом (DESC), покажи топ 5.

In [ ]:
# Спробуй: Фільми 2015-2020 за рейтингом
query = """

"""

# result = pd.read_sql(query, conn)
# print(result)

## Розділ 2: LEFT JOIN

**LEFT JOIN** повертає **всі** рядки з лівої таблиці, навіть якщо немає співпадіння в правій. Для рядків без співпадіння — значення NULL.

**Різниця від INNER JOIN:**
- INNER JOIN: Тільки рядки що співпадають
- LEFT JOIN: Всі рядки з лівої таблиці + співпадіння з правої

In [ ]:
# Порівнюємо INNER vs LEFT JOIN
# INNER: тільки фільми з рейтингами
queryInner = """
SELECT COUNT(DISTINCT tb.tconst) as movieCount
FROM title_basics tb
INNER JOIN title_ratings tr ON tb.tconst = tr.tconst
WHERE tb.titleType = 'movie'
"""
innerResult = pd.read_sql(queryInner, conn)
print(f"Фільми З рейтингами (INNER): {innerResult['movieCount'][0]}")

# LEFT: всі фільми (з рейтингами та без)
queryLeft = """
SELECT COUNT(DISTINCT tb.tconst) as movieCount
FROM title_basics tb
LEFT JOIN title_ratings tr ON tb.tconst = tr.tconst
WHERE tb.titleType = 'movie'
"""
leftResult = pd.read_sql(queryLeft, conn)
print(f"ВСІ фільми (LEFT): {leftResult['movieCount'][0]}")

In [ ]:
# Знаходимо фільми БЕЗ рейтингів (NULL значення)
query = """
SELECT 
    tb.primaryTitle,
    tb.startYear,
    tr.averageRating
FROM title_basics tb
LEFT JOIN title_ratings tr ON tb.tconst = tr.tconst
WHERE tb.titleType = 'movie' 
  AND tr.averageRating IS NULL
LIMIT 10
"""

result = pd.read_sql(query, conn)
print("Фільми без рейтингів (NULL):")
print(result)

**Спробуй:** Напиши LEFT JOIN запит щоб знайти серіали (titleType = 'tvSeries') без рейтингів. Покажи назву та рік.

In [ ]:
# Спробуй: Серіали без рейтингів
query = """

"""

# result = pd.read_sql(query, conn)
# print(result)

## Розділ 3: Багатотабличні JOIN

Можна з'єднувати більше ніж 2 таблиці для відповіді на складні запитання.

```sql
SELECT columns
FROM table1
JOIN table2 ON table1.key = table2.key
JOIN table3 ON table2.key = table3.key
WHERE condition
```

In [ ]:
# З'єднуємо 3 таблиці: фільми → актори → імена людей
query = """
SELECT 
    tb.primaryTitle AS movieTitle,
    tb.startYear,
    nb.primaryName AS actorName,
    tp.category AS role
FROM title_basics tb
INNER JOIN title_principals tp ON tb.tconst = tp.tconst
INNER JOIN name_basics nb ON tp.nconst = nb.nconst
WHERE tb.titleType = 'movie' 
  AND tp.category IN ('actor', 'actress')
  AND tb.startYear = 2020
LIMIT 10
"""

result = pd.read_sql(query, conn)
print("Актори у фільмах 2020 року (3 таблиці JOIN):")
print(result)

In [ ]:
# Рахуємо скільки фільмів у кожного актора
query = """
SELECT 
    nb.primaryName AS actorName,
    COUNT(DISTINCT tb.tconst) AS movieCount
FROM name_basics nb
INNER JOIN title_principals tp ON nb.nconst = tp.nconst
INNER JOIN title_basics tb ON tp.tconst = tb.tconst
WHERE tp.category IN ('actor', 'actress')
  AND tb.titleType = 'movie'
GROUP BY nb.nconst, nb.primaryName
ORDER BY movieCount DESC
LIMIT 10
"""

result = pd.read_sql(query, conn)
print("Топ акторів за кількістю фільмів:")
print(result)

In [ ]:
# Складний запит: JOIN + WHERE + GROUP BY + HAVING + ORDER BY
# Середній рейтинг фільмів актора (мінімум 5 фільмів)
query = """
SELECT 
    nb.primaryName AS actorName,
    COUNT(DISTINCT tb.tconst) AS movieCount,
    ROUND(AVG(tr.averageRating), 2) AS avgRating
FROM name_basics nb
INNER JOIN title_principals tp ON nb.nconst = tp.nconst
INNER JOIN title_basics tb ON tp.tconst = tb.tconst
INNER JOIN title_ratings tr ON tb.tconst = tr.tconst
WHERE tp.category IN ('actor', 'actress')
  AND tb.titleType = 'movie'
GROUP BY nb.nconst, nb.primaryName
HAVING COUNT(DISTINCT tb.tconst) >= 5
ORDER BY avgRating DESC
LIMIT 10
"""

result = pd.read_sql(query, conn)
print("Найкращі актори за середнім рейтингом (мін. 5 фільмів):")
print(result)

**Спробуй:** Напиши складний запит з мінімум 2 JOIN, WHERE, GROUP BY. Наприклад: знайди режисерів (category = 'director') з найбільшою кількістю фільмів.

In [ ]:
# Спробуй: Свій складний запит
query = """

"""

# result = pd.read_sql(query, conn)
# print(result)

In [ ]:
# Закриваємо з'єднання
conn.close()
print("З'єднання закрито!")

## Підсумок

Ви навчились три типи SQL JOIN:

1. **INNER JOIN**: Тільки рядки що співпадають в обох таблицях
2. **LEFT JOIN**: Всі рядки з лівої таблиці + співпадіння з правої (NULL для відсутніх)
3. **Багатотабличні JOIN**: З'єднання 3+ таблиць для складного аналізу

В комбінації з GROUP BY, WHERE, HAVING та ORDER BY — JOIN дозволяють відповідати на складні запитання з реляційних баз даних.